In [1]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [2]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dinov2 = torch.hub.load(
    repo_or_dir="facebookresearch/dinov2", 
    model='dinov2_vits14'
)
dinov2 = dinov2.to(device)
dinov2.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using device: cpu


Using cache found in C:\Users\Joris/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\Joris/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\Joris/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\Joris/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [ ]:
def extract_features_from_image_crops(csv_path, path_to_dataset, label_column='super_category'):
    """Reads a CSV of bounding boxes, crops images, and extracts DINOv2 features."""
    df = pd.read_csv(csv_path)
    
    features = []
    labels = []
    
    for _, row in df.iterrows():
        img_name = row['image_filename']
        subfolder = img_name[0].upper()
        img_path = os.path.join(path_to_dataset, subfolder, img_name)
        
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}. Skipping.")
            continue
            
        try:
            # Load image
            img = Image.open(img_path).convert('RGB')
            img_width, img_height = img.size
            
            # Convert normalized YOLO coordinates to absolute pixel coordinates
            x_center = row['x_center'] * img_width
            y_center = row['y_center'] * img_height
            box_width = row['width'] * img_width
            box_height = row['height'] * img_height
            
            left = x_center - (box_width / 2)
            top = y_center - (box_height / 2)
            right = x_center + (box_width / 2)
            bottom = y_center + (box_height / 2)
            
            # Crop the image to the bounding box
            crop_img = img.crop((left, top, right, bottom))
            
            # Prepare tensor and extract features
            img_tensor = transform(crop_img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                feature_vector = dinov2(img_tensor)
                features.append(feature_vector.cpu().numpy().flatten())
                labels.append(row[label_column])
                
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            
    return np.array(features), np.array(labels)